# Advanced Forecasting — Configuring Bayesian Priors

Every Bayesian forecaster in OptiStock ships with a set of default priors that work well on scaled data. For most users those defaults are fine — but when you have **domain knowledge** about a parameter (e.g. you know your weekly seasonality is mild, or your trend is essentially flat), tightening or relaxing the relevant prior usually improves both convergence and forecast calibration.

This notebook shows how to:

1. **Inspect** the priors a model uses, via `model.describe_priors()`.
2. **Tune** a single prior's parameters (e.g. make the observation noise tighter).
3. **Swap** a prior's distribution entirely (e.g. replace a `Normal` with a `Laplace` for sparsity).
4. **Work with grouped priors** in `UnivariateSSM`, where one config field controls every parameter in a *family*.

No model is fit — every cell is fast. The goal is to make the API discoverable.

## The design in one minute

Two pieces:

- **`Prior`** — a lightweight wrapper around a PyMC distribution: `Prior(distribution, params, description)`. Examples: `Prior("Normal", {"mu": 0.0, "sigma": 1.0})`, `Prior("HalfNormal", {"sigma": 0.5})`.
- **`*Priors`** — a dataclass per model whose fields are `Prior` objects. Each field defaults to the value previously hard-coded in the model. Pass an instance to the model's constructor to override; pass nothing to keep defaults.

Every model inherits `describe_priors()` from `BaseForecaster`. It prints a readable table **and** returns a JSON-serializable dict, so you can use it either interactively or programmatically.

In [1]:
import numpy as np
import pandas as pd

from optistock.forecasting import (
    # Models
    BayesTimeSeries,
    BARTBayesTimeSeries,
    HSGPBayesTimeSeries,
    UnivariateSSM,
    # Priors
    BayesTimeSeriesPriors,
    BARTBayesTimeSeriesPriors,
    HSGPBayesTimeSeriesPriors,
    UnivariateSSMPriors,
    # Prior wrapper
    Prior,
)

# A tiny dummy dataset is enough to instantiate the models — we never call .fit()
df = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=30, freq="D"),
    "sales": np.arange(30, dtype=float) + 10,
})

## 1. `BayesTimeSeries` — Fourier regressor

Five priors: `intercept`, `growth`, `beta_event`, `beta_fourier`, `sigma`. All values live in **scaled `[0, 1]` space** — the model min-max-scales the target before fitting, so the defaults assume that scale. If you write your own priors, keep them in that scale too.

In [2]:
model = BayesTimeSeries(df)
model.describe_priors()

BayesTimeSeriesPriors:
Variable      Distribution  Parameters         Description
----------------------------------------------------------
intercept     HalfNormal    sigma=1.0          Baseline level of the series
growth        Normal        mu=0.0, sigma=1.0  Linear trend slope
beta_event    Normal        mu=0.0, sigma=0.5  Per-event additive effect
beta_fourier  Laplace       mu=0.0, b=1.0      Fourier seasonality coefficients
sigma         HalfNormal    sigma=0.05         Observation noise


{'intercept': {'distribution': 'HalfNormal',
  'params': {'sigma': 1.0},
  'description': 'Baseline level of the series'},
 'growth': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 1.0},
  'description': 'Linear trend slope'},
 'beta_event': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 0.5},
  'description': 'Per-event additive effect'},
 'beta_fourier': {'distribution': 'Laplace',
  'params': {'mu': 0.0, 'b': 1.0},
  'description': 'Fourier seasonality coefficients'},
 'sigma': {'distribution': 'HalfNormal',
  'params': {'sigma': 0.05},
  'description': 'Observation noise'}}

### Tighten a single prior

Suppose you have very clean data and want a tighter observation noise prior. Build a new `BayesTimeSeriesPriors` with just that field overridden — the rest stay at their defaults:

In [3]:
tight_noise = BayesTimeSeriesPriors(
    sigma=Prior("HalfNormal", {"sigma": 0.01}, "Tight obs noise — clean data"),
)

model = BayesTimeSeries(df, priors=tight_noise)
model.describe_priors()

BayesTimeSeriesPriors:
Variable      Distribution  Parameters         Description
----------------------------------------------------------
intercept     HalfNormal    sigma=1.0          Baseline level of the series
growth        Normal        mu=0.0, sigma=1.0  Linear trend slope
beta_event    Normal        mu=0.0, sigma=0.5  Per-event additive effect
beta_fourier  Laplace       mu=0.0, b=1.0      Fourier seasonality coefficients
sigma         HalfNormal    sigma=0.01         Tight obs noise — clean data


{'intercept': {'distribution': 'HalfNormal',
  'params': {'sigma': 1.0},
  'description': 'Baseline level of the series'},
 'growth': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 1.0},
  'description': 'Linear trend slope'},
 'beta_event': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 0.5},
  'description': 'Per-event additive effect'},
 'beta_fourier': {'distribution': 'Laplace',
  'params': {'mu': 0.0, 'b': 1.0},
  'description': 'Fourier seasonality coefficients'},
 'sigma': {'distribution': 'HalfNormal',
  'params': {'sigma': 0.01},
  'description': 'Tight obs noise — clean data'}}

### Swap the distribution

The default `beta_fourier` prior is `Laplace` (encourages sparsity in seasonal harmonics). If you want to use a Gaussian instead — perhaps because you believe all your harmonics are non-zero and similar in size — just swap the distribution name and provide the right parameters:

In [4]:
gaussian_seasonal = BayesTimeSeriesPriors(
    beta_fourier=Prior("Normal", {"mu": 0.0, "sigma": 0.5}, "Gaussian seasonality"),
)

model = BayesTimeSeries(df, priors=gaussian_seasonal)
model.describe_priors();

BayesTimeSeriesPriors:
Variable      Distribution  Parameters         Description
----------------------------------------------------------
intercept     HalfNormal    sigma=1.0          Baseline level of the series
growth        Normal        mu=0.0, sigma=1.0  Linear trend slope
beta_event    Normal        mu=0.0, sigma=0.5  Per-event additive effect
beta_fourier  Normal        mu=0.0, sigma=0.5  Gaussian seasonality
sigma         HalfNormal    sigma=0.05         Observation noise


## 2. `BARTBayesTimeSeries` — Bayesian Additive Regression Trees

Only three priors here: `intercept`, `growth`, `sigma`. The BART component itself is non-parametric and is controlled by the `trees` constructor argument, not a prior.

In [5]:
model = BARTBayesTimeSeries(df)
model.describe_priors()

BARTBayesTimeSeriesPriors:
Variable   Distribution  Parameters         Description
-------------------------------------------------------
intercept  HalfNormal    sigma=1.0          Baseline level of the series
growth     Normal        mu=0.0, sigma=1.0  Linear trend slope
sigma      HalfNormal    sigma=0.1          Observation noise


{'intercept': {'distribution': 'HalfNormal',
  'params': {'sigma': 1.0},
  'description': 'Baseline level of the series'},
 'growth': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 1.0},
  'description': 'Linear trend slope'},
 'sigma': {'distribution': 'HalfNormal',
  'params': {'sigma': 0.1},
  'description': 'Observation noise'}}

In [6]:
# Example: you expect a strong upward drift; widen the growth prior
custom = BARTBayesTimeSeriesPriors(
    growth=Prior("Normal", {"mu": 0.5, "sigma": 2.0}, "Expected strong upward drift"),
)
BARTBayesTimeSeries(df, priors=custom).describe_priors()

BARTBayesTimeSeriesPriors:
Variable   Distribution  Parameters         Description
-------------------------------------------------------
intercept  HalfNormal    sigma=1.0          Baseline level of the series
growth     Normal        mu=0.5, sigma=2.0  Expected strong upward drift
sigma      HalfNormal    sigma=0.1          Observation noise


{'intercept': {'distribution': 'HalfNormal',
  'params': {'sigma': 1.0},
  'description': 'Baseline level of the series'},
 'growth': {'distribution': 'Normal',
  'params': {'mu': 0.5, 'sigma': 2.0},
  'description': 'Expected strong upward drift'},
 'sigma': {'distribution': 'HalfNormal',
  'params': {'sigma': 0.1},
  'description': 'Observation noise'}}

## 3. `HSGPBayesTimeSeries` — Hilbert Space GP approximation

Four priors: `ell` (GP lengthscale), `eta` (amplitude), `intercept`, `sigma`.

**One subtlety:** the default `intercept` prior intentionally omits `mu`. At fit time, the model centres `mu` on `y_scaled.mean()` automatically — this matched the original hard-coded behaviour. If you provide an `intercept` prior with `mu` set, the model uses your value verbatim and skips the auto-centring.

In [8]:
model = HSGPBayesTimeSeries(df)
model.describe_priors()

HSGPBayesTimeSeriesPriors:
Variable   Distribution  Parameters         Description
-------------------------------------------------------
ell        InverseGamma  mu=0.5, sigma=0.2  GP lengthscale
eta        Exponential   lam=1.0            GP amplitude
intercept  Normal        sigma=0.5          Baseline level. If `mu` is not provided, fit() injects y_scaled.mean().
sigma      HalfNormal    sigma=0.1          Observation noise


{'ell': {'distribution': 'InverseGamma',
  'params': {'mu': 0.5, 'sigma': 0.2},
  'description': 'GP lengthscale'},
 'eta': {'distribution': 'Exponential',
  'params': {'lam': 1.0},
  'description': 'GP amplitude'},
 'intercept': {'distribution': 'Normal',
  'params': {'sigma': 0.5},
  'description': 'Baseline level. If `mu` is not provided, fit() injects y_scaled.mean().'},
 'sigma': {'distribution': 'HalfNormal',
  'params': {'sigma': 0.1},
  'description': 'Observation noise'}}

In [9]:
# Example: pin a smoother GP by widening the lengthscale prior, and set an explicit intercept
smooth_gp = HSGPBayesTimeSeriesPriors(
    ell=Prior("InverseGamma", {"mu": 1.0, "sigma": 0.3}, "Longer lengthscale -> smoother GP"),
    intercept=Prior("Normal", {"mu": 0.5, "sigma": 0.2}, "Explicit centre, skips auto y.mean()"),
)
HSGPBayesTimeSeries(df, priors=smooth_gp).describe_priors();

HSGPBayesTimeSeriesPriors:
Variable   Distribution  Parameters         Description
-------------------------------------------------------
ell        InverseGamma  mu=1.0, sigma=0.3  Longer lengthscale -> smoother GP
eta        Exponential   lam=1.0            GP amplitude
intercept  Normal        mu=0.5, sigma=0.2  Explicit centre, skips auto y.mean()
sigma      HalfNormal    sigma=0.1          Observation noise


## 4. `UnivariateSSM` — priors grouped by family

The state space model has a **variable number** of priors at runtime — one per latent state, one per exogenous regressor, plus seasonals. Listing each one individually would mean the dataclass changes shape with `build_model()` arguments.

Instead, priors are **grouped by family**. Every parameter in a family shares the same prior. This matches the dynamic logic in `_register_priors()` and lets you configure the model before you've even decided which components it contains.

| Field | Applies to | PyMC variables it covers |
|---|---|---|
| `initial_state_cov` | Diagonal scale of initial state covariance | `P0_diag` |
| `initial_state` | Initial state values | `initial_level_trend`, `initial_<regressor>`, ... |
| `observation_noise` | Measurement noise | `sigma_obs` |
| `regression_beta` | Regression coefficient magnitudes | `beta_<regressor>` |
| `regression_innovation` | Innovation variance for time-varying regression coefs | `sigma_beta_<regressor>` |
| `process_noise` | Process noise (level / slope) | `sigma_level_trend`, ... |
| `seasonal_amplitude` | Initial seasonal amplitudes | `params_seasonal` |
| `seasonal_innovation` | Process noise for seasonal amplitudes | `sigma_seasonal` |

In [10]:
model = UnivariateSSM(df.set_index("date"), target_col="sales")
model.describe_priors()

UnivariateSSMPriors:
Variable               Distribution  Parameters         Description
-------------------------------------------------------------------
initial_state_cov      Gamma         alpha=2, beta=10   Diagonal scale of the initial state covariance (P0_diag)
initial_state          Normal        mu=0.5, sigma=1.0  Initial state values (initial_*)
observation_noise      HalfNormal    sigma=0.05         Measurement noise (sigma_obs)
regression_beta        HalfNormal    sigma=3.0          Regression coefficient magnitudes (beta_*)
regression_innovation  Gamma         alpha=2, beta=50   Innovation variance for time-varying regression coefs (sigma_beta_*)
process_noise          Gamma         alpha=2, beta=50   Process noise for trend / level / slope (sigma_*)
seasonal_amplitude     Normal        mu=0.0, sigma=0.5  Initial seasonal amplitudes (params_*)
seasonal_innovation    Gamma         alpha=2, beta=50   Process noise for seasonal amplitudes (sigma_seasonal)


{'initial_state_cov': {'distribution': 'Gamma',
  'params': {'alpha': 2, 'beta': 10},
  'description': 'Diagonal scale of the initial state covariance (P0_diag)'},
 'initial_state': {'distribution': 'Normal',
  'params': {'mu': 0.5, 'sigma': 1.0},
  'description': 'Initial state values (initial_*)'},
 'observation_noise': {'distribution': 'HalfNormal',
  'params': {'sigma': 0.05},
  'description': 'Measurement noise (sigma_obs)'},
 'regression_beta': {'distribution': 'HalfNormal',
  'params': {'sigma': 3.0},
  'description': 'Regression coefficient magnitudes (beta_*)'},
 'regression_innovation': {'distribution': 'Gamma',
  'params': {'alpha': 2, 'beta': 50},
  'description': 'Innovation variance for time-varying regression coefs (sigma_beta_*)'},
 'process_noise': {'distribution': 'Gamma',
  'params': {'alpha': 2, 'beta': 50},
  'description': 'Process noise for trend / level / slope (sigma_*)'},
 'seasonal_amplitude': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 0.5},


### Example: a more conservative SSM

Suppose you want the latent states to evolve **slowly** (less drift between time steps). That means tighter `process_noise` and `seasonal_innovation` priors. We'll also tighten the observation noise:

In [11]:
conservative = UnivariateSSMPriors(
    process_noise=Prior("Gamma", {"alpha": 2, "beta": 200}, "Slow trend evolution"),
    seasonal_innovation=Prior("Gamma", {"alpha": 2, "beta": 200}, "Slow seasonal evolution"),
    observation_noise=Prior("HalfNormal", {"sigma": 0.02}, "Tight obs noise"),
)
UnivariateSSM(df.set_index("date"), target_col="sales", priors=conservative).describe_priors()

UnivariateSSMPriors:
Variable               Distribution  Parameters         Description
-------------------------------------------------------------------
initial_state_cov      Gamma         alpha=2, beta=10   Diagonal scale of the initial state covariance (P0_diag)
initial_state          Normal        mu=0.5, sigma=1.0  Initial state values (initial_*)
observation_noise      HalfNormal    sigma=0.02         Tight obs noise
regression_beta        HalfNormal    sigma=3.0          Regression coefficient magnitudes (beta_*)
regression_innovation  Gamma         alpha=2, beta=50   Innovation variance for time-varying regression coefs (sigma_beta_*)
process_noise          Gamma         alpha=2, beta=200  Slow trend evolution
seasonal_amplitude     Normal        mu=0.0, sigma=0.5  Initial seasonal amplitudes (params_*)
seasonal_innovation    Gamma         alpha=2, beta=200  Slow seasonal evolution


{'initial_state_cov': {'distribution': 'Gamma',
  'params': {'alpha': 2, 'beta': 10},
  'description': 'Diagonal scale of the initial state covariance (P0_diag)'},
 'initial_state': {'distribution': 'Normal',
  'params': {'mu': 0.5, 'sigma': 1.0},
  'description': 'Initial state values (initial_*)'},
 'observation_noise': {'distribution': 'HalfNormal',
  'params': {'sigma': 0.02},
  'description': 'Tight obs noise'},
 'regression_beta': {'distribution': 'HalfNormal',
  'params': {'sigma': 3.0},
  'description': 'Regression coefficient magnitudes (beta_*)'},
 'regression_innovation': {'distribution': 'Gamma',
  'params': {'alpha': 2, 'beta': 50},
  'description': 'Innovation variance for time-varying regression coefs (sigma_beta_*)'},
 'process_noise': {'distribution': 'Gamma',
  'params': {'alpha': 2, 'beta': 200},
  'description': 'Slow trend evolution'},
 'seasonal_amplitude': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 0.5},
  'description': 'Initial seasonal amplitu

## Programmatic introspection

`describe_priors()` also **returns** a structured dict (in addition to printing). You can use it to log prior configurations alongside experiment runs, persist them to JSON, or diff two configurations:

In [12]:
import json

model = BayesTimeSeries(df)
prior_spec = model.describe_priors()    # prints + returns the dict

print("\nJSON dump:")
print(json.dumps(prior_spec, indent=2))

BayesTimeSeriesPriors:
Variable      Distribution  Parameters         Description
----------------------------------------------------------
intercept     HalfNormal    sigma=1.0          Baseline level of the series
growth        Normal        mu=0.0, sigma=1.0  Linear trend slope
beta_event    Normal        mu=0.0, sigma=0.5  Per-event additive effect
beta_fourier  Laplace       mu=0.0, b=1.0      Fourier seasonality coefficients
sigma         HalfNormal    sigma=0.05         Observation noise

JSON dump:
{
  "intercept": {
    "distribution": "HalfNormal",
    "params": {
      "sigma": 1.0
    },
    "description": "Baseline level of the series"
  },
  "growth": {
    "distribution": "Normal",
    "params": {
      "mu": 0.0,
      "sigma": 1.0
    },
    "description": "Linear trend slope"
  },
  "beta_event": {
    "distribution": "Normal",
    "params": {
      "mu": 0.0,
      "sigma": 0.5
    },
    "description": "Per-event additive effect"
  },
  "beta_fourier": {
    "distr

In [13]:
# Or call to_dict() directly on the priors object, without printing
BayesTimeSeriesPriors().to_dict()

{'intercept': {'distribution': 'HalfNormal',
  'params': {'sigma': 1.0},
  'description': 'Baseline level of the series'},
 'growth': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 1.0},
  'description': 'Linear trend slope'},
 'beta_event': {'distribution': 'Normal',
  'params': {'mu': 0.0, 'sigma': 0.5},
  'description': 'Per-event additive effect'},
 'beta_fourier': {'distribution': 'Laplace',
  'params': {'mu': 0.0, 'b': 1.0},
  'description': 'Fourier seasonality coefficients'},
 'sigma': {'distribution': 'HalfNormal',
  'params': {'sigma': 0.05},
  'description': 'Observation noise'}}

## Summary

- Instantiate any model with `priors=ModelPriors(...)` to override defaults; pass nothing to keep them.
- Each prior is a `Prior(distribution, params, description)` — distribution is the PyMC class name, `params` are its kwargs.
- Use `model.describe_priors()` to see what's configured and what each prior controls.
- `UnivariateSSM` groups priors by **family**, not per-variable — one field controls every member of that family.
- Defaults are tuned for **scaled `[0, 1]` target data** (the models min-max-scale internally). Custom priors should respect that scale.